# VQE for H₂ — Ground State Energy Estimation

**Variational Quantum Eigensolver (VQE)** with UCCSD ansatz for the hydrogen molecule.
Implements standard VQE, ADAPT-VQE, optimizer comparison, shot noise, and noisy simulation.

## Step 1 — Imports

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
np.set_printoptions(precision=6, suppress=True)

In [2]:
from qiskit_algorithms import VQE, AdaptVQE
from qiskit_algorithms.optimizers import COBYLA, SPSA, L_BFGS_B

from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD
from qiskit_nature.second_q.circuit.library.initial_states import HartreeFock

# from qiskit.primitives import Estimator, Sampler
from qiskit_aer import AerSimulator
# from qiskit_aer.noise import NoiseModel, thermal_noise_model

from qiskit_ibm_runtime import QiskitRuntimeService

print("All imports OK")

All imports OK


In [3]:
from qiskit.quantum_info import SparsePauliOp

hamiltonian = SparsePauliOp(
    [
        "IIII",
        "IIIZ",
        "IZII",
        "IIZI",
        "ZIII",
        "IZIZ",
        "IIZZ",
        "ZIIZ",
        "IZZI",
        "ZZII",
        "ZIZI",
        "YYYY",
        "XXYY",
        "YYXX",
        "XXXX",
    ],
    coeffs=[
        -0.09820182 + 0.0j,
        -0.1740751 + 0.0j,
        -0.1740751 + 0.0j,
        0.2242933 + 0.0j,
        0.2242933 + 0.0j,
        0.16891402 + 0.0j,
        0.1210099 + 0.0j,
        0.16631441 + 0.0j,
        0.16631441 + 0.0j,
        0.1210099 + 0.0j,
        0.17504456 + 0.0j,
        0.04530451 + 0.0j,
        0.04530451 + 0.0j,
        0.04530451 + 0.0j,
        0.04530451 + 0.0j,
    ],
)

import numpy as np

A = np.array(hamiltonian)
eigenvalues, eigenvectors = np.linalg.eigh(A)
print("The ground state energy is ", min(eigenvalues), "using hartfree fock method with brute force solution")

The ground state energy is  -1.1459778447627311 using hartfree fock method with brute force solution


## Step 2 — H₂ Molecular Hamiltonian

PySCF driver computes the electronic structure at equilibrium bond length (~0.735 Å).
STO-3G basis set, Hartree-Fock reference, Jordan-Wigner fermion-to-qubit mapping.

In [6]:
molecule = """H 0.0 0.0 0.0
H 0.735 0.0 0.0
"""

driver = PySCFDriver(atom=molecule, basis="sto3g")
problem = driver.run()

num_spatial_orbitals = problem.num_spatial_orbitals
num_particles = problem.num_particles
nuclear_repulsion = problem.nuclear_repulsion_energy

hamiltonian = problem.hamiltonian
second_q_op = hamiltonian.second_q_op()
mapper = JordanWignerMapper()
qubit_op = mapper.map(second_q_op)

print(f"Spatial orbitals: {num_spatial_orbitals}")
print(f"Spin orbitals:     {problem.num_spin_orbitals}")
print(f"Electrons:        {num_particles}")
print(f"Qubits:           {qubit_op.num_qubits}")
print(f"Nuclear repulsion: {nuclear_repulsion:.6f} Ha")

Spatial orbitals: 2
Spin orbitals:     4
Electrons:        (1, 1)
Qubits:           4
Nuclear repulsion: 0.719969 Ha


## Step 3 — Exact Classical Ground State Energy

Diagonalize the qubit Hamiltonian for the exact reference value.

In [7]:
exact_matrix = qubit_op.to_matrix()
eigenvalues, _ = np.linalg.eigh(exact_matrix)
exact_energy = eigenvalues[0] + nuclear_repulsion

print(f"Exact ground state energy: {exact_energy:.12f} Ha")
print(f"Literature reference:     ~-1.137270 Ha")

Exact ground state energy: -1.137306035753 Ha
Literature reference:     ~-1.137270 Ha


## Step 4 — Hartree-Fock Reference & UCCSD Ansatz

In [8]:
initial_state = HartreeFock(
    num_spatial_orbitals=num_spatial_orbitals,
    num_particles=num_particles,
    qubit_mapper=mapper,
)

ansatz = UCCSD(
    num_spatial_orbitals=num_spatial_orbitals,
    num_particles=num_particles,
    qubit_mapper=mapper,
    initial_state=initial_state,
)

print(f"Ansatz parameters: {ansatz.num_parameters}")
print(f"Number of qubits:  {ansatz.num_qubits}")
print(f"Circuit depth:     {ansatz.depth()}")

Ansatz parameters: 3
Number of qubits:  4
Circuit depth:     1


## Step 5 — VQE with COBYLA Optimizer (Statevector Simulator)

In [36]:
from qiskit.primitives import StatevectorEstimator
# Use StatevectorEstimator for ideal local simulation
estimator = StatevectorEstimator()
# Create a list to store the history
cobyla_history_raw = []
def callback_ideal(eval_count, parameters, mean, metadata):
    cobyla_history_raw.append(mean)

vqe_cobyla = VQE(
    estimator=estimator,
    ansatz=ansatz,
    optimizer=COBYLA(maxiter=500),
    callback=callback_ideal  # <--- Add the callback here
)

print("Running VQE with COBYLA...")
result_cobyla = vqe_cobyla.compute_minimum_eigenvalue(qubit_op)
energy_cobyla = result_cobyla.eigenvalue.real + nuclear_repulsion

error_cobyla = abs(energy_cobyla - exact_energy)

print(f"\nVQE (COBYLA) energy: {energy_cobyla:.12f} Ha")
print(f"Exact energy:           {exact_energy:.12f} Ha")
print(f"Error:                 {error_cobyla:.12f} Ha ({100*error_cobyla/abs(exact_energy):.4f}%)")
print(f"Iterations:             {result_cobyla.optimizer_evals}")

Running VQE with COBYLA...

VQE (COBYLA) energy: -1.137306030234 Ha
Exact energy:           -1.137306035753 Ha
Error:                 0.000000005519 Ha (0.0000%)
Iterations:             None


## Step 6 — Optimizer Comparison: COBYLA vs SPSA vs L-BFGS-B

We compare three classical optimizers on the same UCCSD ansatz and Hamiltonian.
SPSA is designed for noisy hardware evaluation. L-BFGS-B uses gradient approximations.

In [14]:
optimizers_config = [
    ("COBYLA", COBYLA(maxiter=500)),
    ("SPSA", SPSA(maxiter=500)),
    ("L-BFGS-B", L_BFGS_B(maxiter=500)),
]

optimizer_results = {}

print("Running optimizer comparison...")
for name, optimizer in optimizers_config:
    # Capture history using a callback (standard in V2/Algorithms)
    history = []
    def callback(eval_count, parameters, mean, metadata):
        # mean is the expectation value (eigenvalue) at this step
        history.append(mean)

    # Ensure 'estimator' is the StatevectorEstimator defined earlier
    vqe = VQE(
        estimator=estimator, 
        ansatz=ansatz, 
        optimizer=optimizer, 
        callback=callback
    )
    
    result = vqe.compute_minimum_eigenvalue(qubit_op)
    
    energy = result.eigenvalue.real + nuclear_repulsion
    # Add nuclear repulsion to the captured history for plotting
    energy_history = [ev + nuclear_repulsion for ev in history]
    
    optimizer_results[name] = {
        "energy": energy,
        "error": abs(energy - exact_energy),
        "iterations": result.optimizer_evals,
        "history": energy_history,
    }
    print(f"  {name}: energy={energy:.8f}, error={abs(energy-exact_energy):.2e}, iters={result.optimizer_evals}")


Running optimizer comparison...
  COBYLA: energy=-1.13730603, error=4.40e-09, iters=None
  SPSA: energy=-1.13725574, error=5.03e-05, iters=None
  L-BFGS-B: energy=-1.13730604, error=2.12e-11, iters=None


## Step 7 — Optimizer Convergence Plot

In [16]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = ["tab:blue", "tab:orange", "tab:green"]
markers = ["-", "--", ":"]

for (name, res), color, mkr in zip(optimizer_results.items(), colors, markers):
    ax.plot(res["history"], label=name, color=color, linewidth=1.8,
            linestyle=mkr)

ax.axhline(y=exact_energy, color="red", linestyle="--", linewidth=2,
           label=f"Exact ({exact_energy:.6f} Ha)")

ax.set_xlabel("Optimizer Iteration", fontsize=12)
ax.set_ylabel("Energy (Hartree)", fontsize=12)
ax.set_title("VQE Optimizer Comparison for H₂ Ground State\n(UCCSD Ansatz)", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("vqe_h2/optimizer_comparison.png", dpi=150)
plt.show()

## Step 8 — ADAPT-VQE (Adaptive Operator Pool)

ADAPT-VQE (Grimsley et al., 2018) builds the ansatz iteratively:
1. Compute gradients of all pool operators w.r.t. current state
2. Add the operator with largest gradient to the ansatz
3. Re-optimize variational parameters
4. Repeat until gradients fall below threshold (1e-6)

Result: sparser ansatz, fewer parameters, potentially better convergence.

In [20]:
# Map excitation operators to qubit operators
pool_ops = [mapper.map(op) for op in ansatz.excitation_ops()]

vqe_solver = VQE(
    estimator=estimator,
    ansatz=None, 
    optimizer=COBYLA(maxiter=500)
)

adapt_vqe = AdaptVQE(
    solver=vqe_solver,
    operators=pool_ops,
    gradient_threshold=1e-6,
    initial_state=initial_state,  # <--- CRITICAL: Pass the Hartree-Fock state here
)

print("Running ADAPT-VQE...")
# This should now find non-zero gradients and proceed
result_adapt = adapt_vqe.compute_minimum_eigenvalue(qubit_op)


energy_adapt = result_adapt.eigenvalue.real + nuclear_repulsion
error_adapt = abs(energy_adapt - exact_energy)

# Capture the number of steps (operators added) from the result history
adapt_steps = len(result_adapt.eigenvalue_history)

print(f"\nADAPT-VQE energy: {energy_adapt:.12f} Ha")
print(f"Exact energy:       {exact_energy:.12f} Ha")
print(f"Error:              {error_adapt:.12f} Ha")
print(f"Operators added:    {adapt_steps}")
print(f"UCCSD parameters:   {ansatz.num_parameters}")
print(f"Sparsity gain:      {100*(1 - adapt_steps/ansatz.num_parameters):.1f}%")


Running ADAPT-VQE...

ADAPT-VQE energy: -1.137306034134 Ha
Exact energy:       -1.137306035753 Ha
Error:              0.000000001620 Ha
Operators added:    1
UCCSD parameters:   3
Sparsity gain:      66.7%


## Step 9 — ADAPT-VQE Convergence Plot

In [21]:
adapt_history = [
    ev + nuclear_repulsion
    for ev in result_adapt.eigenvalue_history
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(adapt_history, "b-o", markersize=5, label="ADAPT-VQE")
axes[0].axhline(y=exact_energy, color="red", linestyle="--", linewidth=1.5,
                label=f"Exact ({exact_energy:.6f} Ha)")
axes[0].set_xlabel("ADAPT Iteration", fontsize=11)
axes[0].set_ylabel("Energy (Hartree)", fontsize=11)
axes[0].set_title("ADAPT-VQE Energy Convergence", fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(["ADAPT-VQE", "UCCSD-VQE"],
           [adapt_steps, ansatz.num_parameters],
           color=["tab:blue", "tab:green"])
axes[1].set_ylabel("Number of Parameters", fontsize=11)
axes[1].set_title("Ansatz Sparsity Comparison", fontsize=13)
for i, v in enumerate([adapt_steps, ansatz.num_parameters]):
    axes[1].text(i, v + 0.2, str(v), ha="center", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig("vqe_h2/adapt_comparison.png", dpi=150)
plt.show()

## Step 10 — Shot Noise Analysis

Real hardware uses finite shots (samples), introducing stochastic noise.
We simulate this by running VQE with the Sampler primitive using finite shots.
Repeating the run multiple times shows the variance in energy estimation.

In [28]:
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2

# 1. Transpile the ansatz for the Aer backend
backend = AerSimulator()
ansatz_opt = transpile(ansatz, backend=backend)

shot_counts = [1024, 4096, 8192]
num_repeats = 10
shot_results = {}

print("Running shot noise analysis...")
for shots in shot_counts:
    # 2. Correct EstimatorV2 initialization
    loop_estimator = EstimatorV2(
        options={
            "run_options": {"shots": shots}
        }
    )
    
    vqe = VQE(
        estimator=loop_estimator, 
        ansatz=ansatz_opt,
        optimizer=COBYLA(maxiter=300)
    )
    
    energies = []
    for rep in range(num_repeats):
        result = vqe.compute_minimum_eigenvalue(qubit_op)
        energies.append(result.eigenvalue.real + nuclear_repulsion)
        
    shot_results[shots] = {
        "mean": np.mean(energies),
        "std": np.std(energies),
        "min": np.min(energies),
        "max": np.max(energies),
    }
    print(f"  Shots={shots:5d}: mean={np.mean(energies):.8f}, std={np.std(energies):.8f}")


Running shot noise analysis...
  Shots= 1024: mean=-1.13730603, std=0.00000001
  Shots= 4096: mean=-1.13730603, std=0.00000000
  Shots= 8192: mean=-1.13730603, std=0.00000000


## Step 11 — Shot Noise Variance Plot

In [32]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

shots_list = list(shot_results.keys())
stds = [shot_results[s]["std"] for s in shots_list]
means = [shot_results[s]["mean"] for s in shots_list]

axes[0].errorbar(shots_list, means, yerr=stds, fmt="bo-", capsize=6, linewidth=2, markersize=8, label="Finite shots")
# Fixed typo: 'abel' -> 'label'
axes[0].axhline(y=exact_energy, color="red", linestyle="--", linewidth=2, label=f"Exact ({exact_energy:.6f} Ha)")
axes[0].set_xscale("log")
axes[0].set_xlabel("Number of Shots", fontsize=12)
axes[0].set_ylabel("Energy (Hartree)", fontsize=12)
axes[0].set_title("Shot Noise: Energy vs Shots", fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(range(len(shots_list)), stds, color="tab:orange")
axes[1].set_xticks(range(len(shots_list)))
axes[1].set_xticklabels([str(s) for s in shots_list])
axes[1].set_xlabel("Number of Shots", fontsize=12)
axes[1].set_ylabel("Energy Std Dev (Hartree)", fontsize=12)
axes[1].set_title("Shot Noise Variance vs Shots", fontsize=13)
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("vqe_h2/shot_noise_analysis.png", dpi=150)
plt.show()

print("Shot noise analysis complete. Variance decreases with more shots.")


Shot noise analysis complete. Variance decreases with more shots.


## Step 12 — Noisy Aer Simulation (Hardware Error Model)

We simulate real IBM hardware noise using Aer's thermal noise model.
This captures decoherence (T1/T2), gate errors, and readout errors.
Compare against the ideal statevector result to quantify hardware impact.

In [37]:
from qiskit_aer.noise import NoiseModel
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# 1. Setup Noise Model
# Using AerSimulator() as a baseline for noise parameters
backend = AerSimulator()
noise_model = NoiseModel.from_backend(backend)

# 2. Configure the Noisy Estimator (V2 API)
# The noise model goes in backend_options, shots in run_options
noisy_estimator = EstimatorV2(
    options={
        "backend_options": {"noise_model": noise_model},
        "run_options": {"shots": 8192}
    }
)

# 3. Transpile the ansatz to the Aer basis gates
ansatz_opt = transpile(ansatz, backend=backend)

noisy_history_raw = []
def callback_noisy(eval_count, parameters, mean, metadata):
    noisy_history_raw.append(mean)

vqe_noisy = VQE(
    estimator=noisy_estimator,
    ansatz=ansatz_opt,
    optimizer=COBYLA(maxiter=300),
    callback=callback_noisy  # <--- Add the callback here
)

print("Running VQE with simulated hardware noise...")
result_noisy = vqe_noisy.compute_minimum_eigenvalue(qubit_op)
energy_noisy = result_noisy.eigenvalue.real + nuclear_repulsion

print("Running VQE with simulated hardware noise...")
result_noisy = vqe_noisy.compute_minimum_eigenvalue(qubit_op)

energy_noisy = result_noisy.eigenvalue.real + nuclear_repulsion
error_noisy = abs(energy_noisy - exact_energy)

print(f"\nVQE (noisy simulation) energy: {energy_noisy:.12f} Ha")
print(f"VQE (statevector) energy:      {energy_cobyla:.12f} Ha")
print(f"Exact energy:                  {exact_energy:.12f} Ha")
print(f"Noise-induced error:           {abs(energy_noisy - energy_cobyla):.12f} Ha")


Running VQE with simulated hardware noise...
Running VQE with simulated hardware noise...

VQE (noisy simulation) energy: -1.137306033147 Ha
VQE (statevector) energy:      -1.137306030234 Ha
Exact energy:                  -1.137306035753 Ha
Noise-induced error:           0.000000002913 Ha


## Step 13 — Noisy vs Ideal Convergence Comparison

In [38]:
# Add nuclear repulsion to the captured histories
noisy_history = [ev + nuclear_repulsion for ev in noisy_history_raw]
cobyla_history = [ev + nuclear_repulsion for ev in cobyla_history_raw]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(noisy_history, "r-", linewidth=1.5, alpha=0.8, label="VQE (Noisy Simulation)")
ax.plot(cobyla_history, "b-", linewidth=1.5, alpha=0.8, label="VQE (Ideal Statevector)")
ax.axhline(y=exact_energy, color="green", linestyle="--", linewidth=2,
           label=f"Exact ({exact_energy:.6f} Ha)")

ax.set_xlabel("Optimizer Iteration", fontsize=12)
ax.set_ylabel("Energy (Hartree)", fontsize=12)
ax.set_title("VQE Energy Convergence: Noisy vs Ideal Simulation", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("vqe_h2/noisy_vs_ideal.png", dpi=150)
plt.show()

print(f"Noise-induced energy deviation: {abs(energy_noisy - energy_cobyla):.6f} Ha")


Noise-induced energy deviation: 0.000000 Ha


## Step 14 — Final Results Summary

In [39]:
print("=" * 75)
print(f"{'RESULTS SUMMARY — VQE for H₂ Ground State':^75}")
print("=" * 75)
print(f"Exact classical ground state energy:   {exact_energy:.12f} Ha")
print("-" * 75)
print(f"{'Method':<30} {'Energy (Ha)':<20} {'Error':<20}")
print("-" * 75)
print(f"{'UCCSD-VQE + COBYLA (statevector)':<30} {energy_cobyla:<20.12f} {error_cobyla:<20.12f}")
print(f"{'UCCSD-VQE + SPSA':<30} {optimizer_results['SPSA']['energy']:<20.12f} {optimizer_results['SPSA']['error']:<20.12f}")
print(f"{'UCCSD-VQE + L-BFGS-B':<30} {optimizer_results['L-BFGS-B']['energy']:<20.12f} {optimizer_results['L-BFGS-B']['error']:<20.12f}")
print(f"{'ADAPT-VQE':<30} {energy_adapt:<20.12f} {error_adapt:<20.12f}")
print(f"{'UCCSD-VQE (noisy simulation)':<30} {energy_noisy:<20.12f} {error_noisy:<20.12f}")
print("=" * 75)
print(f"\nADAPT-VQE sparsity: {adapt_steps} vs UCCSD {ansatz.num_parameters} parameters "
      f"({100*(1-adapt_steps/ansatz.num_parameters):.1f}% reduction)")
print(f"Chemical accuracy threshold: 1.6 mHa (0.0016 Ha)")
print(f"Best error (simulator):       {min(error_cobyla, error_adapt):.12f} Ha")

                 RESULTS SUMMARY — VQE for H₂ Ground State                 
Exact classical ground state energy:   -1.137306035753 Ha
---------------------------------------------------------------------------
Method                         Energy (Ha)          Error               
---------------------------------------------------------------------------
UCCSD-VQE + COBYLA (statevector) -1.137306030234      0.000000005519      
UCCSD-VQE + SPSA               -1.137255735250      0.000050300504      
UCCSD-VQE + L-BFGS-B           -1.137306035732      0.000000000021      
ADAPT-VQE                      -1.137306034134      0.000000001620      
UCCSD-VQE (noisy simulation)   -1.137306033147      0.000000002606      

ADAPT-VQE sparsity: 1 vs UCCSD 3 parameters (66.7% reduction)
Chemical accuracy threshold: 1.6 mHa (0.0016 Ha)
Best error (simulator):       0.000000001620 Ha


## Step 15 — Run on Real IBM Quantum Hardware

Uncomment and run after setting up your IBM Quantum Platform account.
See README.md for setup instructions.

```python
# from qiskit_ibm_runtime import QiskitRuntimeService, Estimator as RuntimeEstimator
#
# service = QiskitRuntimeService(channel="ibm_quantum", token="YOUR_TOKEN_HERE")
# backend = service.least_busy(operational=True, simulator=False, min_qubits=4)
# print(f"Using backend: {backend.name}")
#
# runtime_estimator = RuntimeEstimator(options={"resilience_level": 1})
#
# vqe_hw = VQE(
#     estimator=runtime_estimator,
#     ansatz=ansatz,
#     optimizer=COBYLA(maxiter=100),
# )
# result_hw = vqe_hw.compute_minimum_eigenvalue(qubit_op)
#
# energy_hw = result_hw.eigenvalue.real + nuclear_repulsion
# print(f"VQE (hardware) energy: {energy_hw:.12f} Ha")
# print(f"Exact energy:          {exact_energy:.12f} Ha")
# print(f"Hardware error gap:    {abs(energy_hw - exact_energy):.12f} Ha")
```

**Note:** Real hardware results are affected by gate errors, readout errors,
decoherence (T1/T2), and calibration drift — making them noisier than the Aer simulator.